## Fusion duplicates
Fusions were appearing twice in the detail panel. fact_fusions drops breakpoint1/breakpoint2, so the same gene pair detected at different exon junctions collapses into rows that look identical. Cells below establish that the harmonisation restricts to 367 cell lines but doesn't filter rows, then regenerate the table deduped on (ach_id, gene1, gene2, reading_frame) keeping the highest FFPM, with type carried through so read-throughs can be told apart from real fusions. 43,095 to 34,121 rows.

Reads: `5_OmicsFusionFilteredSupplementary.csv`, `fact_fusions.parquet`.
Writes: `fact_fusions_deduped.csv`.

In [1]:
import pandas as pd
raw = pd.read_csv("../../data/gene properties/5_OmicsFusionFilteredSupplementary.csv", low_memory=False)
fus = pd.read_parquet("../../data/parquet/fact_fusions.parquet")

key = ["ach_id", "gene1_ensg", "gene2_ensg"]
dups = fus[fus.duplicated(key, keep=False)]
print("duplicate rows:", len(dups), "of", len(fus))
print(dups.sort_values(key).head(10).to_string())

# do the raw rows behind one of them differ only by breakpoint?
egfr_raw = raw[(raw["ModelID"] == dups.iloc[0]["ach_id"]) &
               (raw["CanonicalFusionName"] == dups.iloc[0]["fusion_name"])]
print(egfr_raw[["breakpoint1", "breakpoint2", "site1", "site2", "type", "FFPM"]].to_string())

duplicate rows: 16857 of 43095
           ach_id                   gene1_ensg                     gene2_ensg gene1_hugo gene2_hugo      fusion_name      ffpm confidence  supporting_reads  split_reads1  split_reads2  discordant_mates reading_frame
21123  ACH-000001   AREL1 (ENSG00000119682.18)      JDP2 (ENSG00000140044.13)      AREL1       JDP2      AREL1--JDP2  0.175402       high                 9             0             4                 5             .
21124  ACH-000001   AREL1 (ENSG00000119682.18)      JDP2 (ENSG00000140044.13)      AREL1       JDP2      AREL1--JDP2  0.136424       high                 7             0             3                 4             .
21256  ACH-000001    BMP7 (ENSG00000101144.13)  RBM38-AS1 (ENSG00000218018.3)       BMP7  RBM38-AS1  BMP7--RBM38-AS1  0.038978        low                 2             0             2                 0  out-of-frame
21257  ACH-000001    BMP7 (ENSG00000101144.13)  RBM38-AS1 (ENSG00000218018.3)       BMP7  RBM38-AS1  BMP7

In [2]:
# fusions: the harmonised table drops breakpoints, so the same gene pair detected at
# different junctions collapses into apparent duplicates. keep the strongest
# detection per (cell line, gene pair, reading frame) and carry `type` through so
# read-through artefacts can be told apart from real fusions.
raw_fus = pd.read_csv("../../data/gene properties/5_OmicsFusionFilteredSupplementary.csv",
                      low_memory=False)
fus_current = pd.read_parquet("../../data/parquet/fact_fusions.parquet")

fus = raw_fus.rename(columns={
    "ModelID": "ach_id",
    "gene1(ENS ID)": "gene1_ensg",
    "gene2(ENS ID)": "gene2_ensg",
    "CanonicalFusionName": "fusion_name",
    "FFPM": "ffpm",
})

fus["gene1_hugo"] = fus["gene1_ensg"].str.extract(r"^([^\s(]+)")
fus["gene2_hugo"] = fus["gene2_ensg"].str.extract(r"^([^\s(]+)")

keep = ["ach_id", "gene1_ensg", "gene2_ensg", "gene1_hugo", "gene2_hugo",
        "fusion_name", "ffpm", "confidence", "reading_frame", "type",
        "split_reads1", "split_reads2", "discordant_mates"]
fus = fus[keep]

# keep the same cell line set as the existing table - the harmonisation restricts
# to cell lines with multi-omics coverage, not all 1,699 in the raw file
fus = fus[fus["ach_id"].isin(set(fus_current["ach_id"]))]
# strongest detection wins where the same pair and frame appears more than once
fus = (fus.sort_values("ffpm", ascending=False)
          .drop_duplicates(["ach_id", "gene1_ensg", "gene2_ensg", "reading_frame"]))

fus.to_csv("../../data/AZ_harmonized_data/fact_fusions_deduped.csv", index=False)
print(fus.shape)
print(fus["type"].value_counts())

(34121, 13)
type
deletion/read-through                 6908
translocation                         6875
duplication                           6099
inversion                             4442
deletion                              2347
deletion/read-through/5'-5'           1231
translocation/5'-5'                   1223
duplication/5'-5'                     1155
duplication/ITD                        655
inversion/3'-3'                        633
deletion/read-through/3'-3'            518
deletion/5'-5'                         489
inversion/5'-5'                        479
duplication/non-canonical_splicing     463
translocation/3'-3'                    379
duplication/3'-3'                      146
deletion/3'-3'                          79
Name: count, dtype: int64


In [3]:
fus_current = pd.read_parquet("../../data/parquet/fact_fusions.parquet")
print("current:", len(fus_current), "| raw:", len(raw_fus), "| mine:", len(fus))

# does the existing table restrict to default entries?
print("\nraw IsDefaultEntryForModel:", raw_fus["IsDefaultEntryForModel"].value_counts().to_dict())
print("raw confidence:", raw_fus["confidence"].value_counts().to_dict())
print("current confidence:", fus_current["confidence"].value_counts().to_dict())

current: 43095 | raw: 184237 | mine: 34121

raw IsDefaultEntryForModel: {'Yes': 178041, 'No': 6196}
raw confidence: {'low': 76217, 'high': 55521, 'medium': 52499}
current confidence: {'low': 16682, 'high': 13500, 'medium': 12913}


In [4]:
fc = fus_current
for key in [["ach_id", "fusion_name"],
            ["ach_id", "gene1_ensg", "gene2_ensg"],
            ["ach_id", "gene1_ensg", "gene2_ensg", "reading_frame"],
            ["ach_id", "gene1_ensg", "gene2_ensg", "reading_frame", "confidence"]]:
    print(len(fc.drop_duplicates(key)), key)

31469 ['ach_id', 'fusion_name']
32344 ['ach_id', 'gene1_ensg', 'gene2_ensg']
34121 ['ach_id', 'gene1_ensg', 'gene2_ensg', 'reading_frame']
36451 ['ach_id', 'gene1_ensg', 'gene2_ensg', 'reading_frame', 'confidence']


In [5]:
raw_r = raw_fus.rename(columns={"ModelID": "ach_id", "CanonicalFusionName": "fusion_name"})
for key in [["ach_id", "fusion_name"],
            ["ach_id", "fusion_name", "reading_frame"],
            ["ach_id", "fusion_name", "reading_frame", "confidence"]]:
    print(len(raw_r.drop_duplicates(key)), key)

135676 ['ach_id', 'fusion_name']
143959 ['ach_id', 'fusion_name', 'reading_frame']
154265 ['ach_id', 'fusion_name', 'reading_frame', 'confidence']


In [6]:
ach = "ACH-000001"
cur = fus_current[fus_current["ach_id"] == ach]
rw  = raw_fus[raw_fus["ModelID"] == ach]
print("current:", len(cur), "| raw:", len(rw))

surviving = set(cur["fusion_name"])
kept = rw[rw["CanonicalFusionName"].isin(surviving)]
dropped = rw[~rw["CanonicalFusionName"].isin(surviving)]

for col in ["confidence", "type", "reading_frame", "IsDefaultEntryForModel"]:
    print(f"\n{col}")
    print("  kept:   ", kept[col].value_counts().head(5).to_dict())
    print("  dropped:", dropped[col].value_counts().head(5).to_dict())

print("\nffpm kept:", kept["FFPM"].describe()[["min","mean","max"]].round(3).to_dict())
print("ffpm dropped:", dropped["FFPM"].describe()[["min","mean","max"]].round(3).to_dict())

current: 199 | raw: 199

confidence
  kept:    {'high': 98, 'low': 54, 'medium': 47}
  dropped: {}

type
  kept:    {'duplication': 95, 'translocation': 30, 'inversion': 27, 'deletion/read-through': 12, "duplication/5'-5'": 10}
  dropped: {}

reading_frame
  kept:    {'.': 87, 'out-of-frame': 67, 'in-frame': 41, 'stop-codon': 4}
  dropped: {}

IsDefaultEntryForModel
  kept:    {'Yes': 199}
  dropped: {}

ffpm kept: {'min': 0.0, 'mean': 0.308, 'max': 5.788}
ffpm dropped: {'min': nan, 'mean': nan, 'max': nan}


In [7]:
print("cell lines - raw:", raw_fus["ModelID"].nunique(),
      "| current:", fus_current["ach_id"].nunique())

cell lines - raw: 1699 | current: 367


In [8]:
# the two cases that showed the problem
for ach, name in [("ACH-000001", "AREL1--JDP2"), ("ACH-000001", "ANK3--CCDC6")]:
    sub = fus[(fus["ach_id"] == ach) & (fus["fusion_name"] == name)]
    print(f"\n{name}: {len(sub)} row(s)")
    print(sub[["ffpm", "confidence", "reading_frame", "type"]].to_string(index=False))


AREL1--JDP2: 1 row(s)
    ffpm confidence reading_frame      type
0.175402       high             . inversion

ANK3--CCDC6: 2 row(s)
    ffpm confidence reading_frame        type
1.305771       high  out-of-frame duplication
0.662630     medium      in-frame duplication
